In [1]:
print("Hello_12345")

Hello_12345


In [2]:
!pip install sqlite3

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://packagefeedproxy.microsoft.io/pypi/simple/


ERROR: Could not find a version that satisfies the requirement sqlite3 (from versions: none)

[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: C:\Users\v-yaalam\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for sqlite3


In [4]:
import sqlite3

conn = sqlite3.connect("database.db")
cursor = conn.cursor()

cursor.execute("CREATE TABLE IF NOT EXISTS users (id INTEGER PRIMARY KEY, name TEXT)")

name = input("Enter a name: ")
cursor.execute("INSERT INTO users (name) VALUES (?)", (name,))
conn.commit()

cursor.execute("SELECT * FROM users")
for row in cursor.fetchall():
    print(row)

conn.close()

(1, 'codex')


In [6]:
import sqlite3

conn = sqlite3.connect("database.db")
cursor = conn.cursor()

cursor.execute("CREATE TABLE IF NOT EXISTS users (id INTEGER PRIMARY KEY, name TEXT)")

name = input("Enter a name: ")
cursor.execute("INSERT INTO users (name) VALUES (?)", (name,))
conn.commit()

cursor.execute("SELECT * FROM users")
for row in cursor.fetchall():
    print(row)

conn.close()

(1, 'codex')
(2, 'Yaswanth ')
(3, 'Automation Engineer')


In [1]:
import sqlite3

conn = sqlite3.connect("database_duplicate.db")
cursor = conn.cursor()

cursor.execute("CREATE TABLE IF NOT EXISTS users (id INTEGER PRIMARY KEY, name TEXT)")

name = input("Enter a name: ")
cursor.execute("INSERT INTO users (name) VALUES (?)", (name,))
conn.commit()

cursor.execute("SELECT * FROM users")
for row in cursor.fetchall():
    print(row)

conn.close()

(1, 'Yaswanth-alam')


# More Python + SQL examples

Your original cells above are unchanged. Start here and run these new cells in order; you do not need to run the older cells first.

`sqlite3` comes with Python, so no `pip install sqlite3` is needed. These examples use only Python's standard library.

We use `:memory:` (a temporary practice database), not your existing `database.db`. Nothing here changes that file. Run Example 1 again whenever you want fresh practice data; then run the following examples in order.

Use `?` placeholders for values instead of building SQL with f-strings. `execute()` runs SQL, `fetchall()` reads the results, and `commit()` saves a transaction within the current database.

## 1. Create a table and insert multiple rows

Like your `users` example, create a connection and cursor. Add more columns and use `executemany()` to insert a Python list of tuples. `NOT NULL` requires a value, and `CHECK` rejects negative salaries.

In [2]:
import sqlite3

practice_conn = sqlite3.connect(":memory:")
practice_conn.execute("PRAGMA foreign_keys = ON")
practice_cursor = practice_conn.cursor()

practice_cursor.execute("""
    CREATE TABLE employees (
        id INTEGER PRIMARY KEY,
        name TEXT NOT NULL,
        department TEXT NOT NULL,
        salary INTEGER NOT NULL CHECK (salary >= 0)
    )
""")

employees = [
    (1, "Alex", "Engineering", 70000),
    (2, "Maya", "Engineering", 80000),
    (3, "Sam", "Support", 50000),
    (4, "Nina", "Sales", 60000),
]
practice_cursor.executemany(
    "INSERT INTO employees (id, name, department, salary) VALUES (?, ?, ?, ?)",
    employees,
)
practice_conn.commit()

practice_cursor.execute("SELECT * FROM employees ORDER BY id")
for row in practice_cursor.fetchall():
    print(row)

(1, 'Alex', 'Engineering', 70000)
(2, 'Maya', 'Engineering', 80000)
(3, 'Sam', 'Support', 50000)
(4, 'Nina', 'Sales', 60000)


## 2. Filter, sort, and limit results

`WHERE` filters rows, `ORDER BY ... DESC` puts the highest salary first, and `LIMIT` restricts the number of results. Try changing the minimum salary or department.

In [3]:
minimum_salary = 60000
department = "Engineering"

practice_cursor.execute("""
    SELECT name, salary
    FROM employees
    WHERE salary >= ? AND department = ?
    ORDER BY salary DESC, id
    LIMIT ?
""", (minimum_salary, department, 2))

for row in practice_cursor.fetchall():
    print(row)

('Maya', 80000)
('Alex', 70000)


## 3. Search names with LIKE

`%` matches any number of characters. Here `A%` finds names starting with A. Try `S` or `N` instead.

In [4]:
starts_with = "A"
practice_cursor.execute(
    "SELECT id, name FROM employees WHERE name LIKE ? ORDER BY id",
    (starts_with + "%",),
)
print(practice_cursor.fetchall())

[(1, 'Alex')]


## 4. Update one employee

`UPDATE` changes existing data. Keep the `WHERE` condition: without it, every employee's salary would change. Setting a fixed salary makes this example safe to rerun.

In [5]:
practice_cursor.execute(
    "UPDATE employees SET salary = ? WHERE id = ?",
    (75000, 1),
)
practice_conn.commit()

practice_cursor.execute("SELECT * FROM employees WHERE id = ?", (1,))
print(practice_cursor.fetchone())

(1, 'Alex', 'Engineering', 75000)


## 5. Delete a practice row

Insert a temporary employee, then delete only that employee by ID. A `DELETE` without `WHERE` would remove every row in the table.

In [6]:
practice_cursor.execute(
    "INSERT INTO employees (id, name, department, salary) VALUES (?, ?, ?, ?)",
    (99, "Temporary Employee", "Support", 40000),
)
practice_cursor.execute("DELETE FROM employees WHERE id = ?", (99,))
print("Rows deleted:", practice_cursor.rowcount)
practice_conn.commit()

practice_cursor.execute("SELECT COUNT(*) FROM employees")
print("Employees remaining:", practice_cursor.fetchone()[0])

Rows deleted: 1
Employees remaining: 4


## 6. Build a department report

`COUNT`, `AVG`, and `SUM` summarize data. `GROUP BY` creates one result per department. `HAVING` filters those groups; here it keeps departments with at least two employees.

In [7]:
practice_cursor.execute("""
    SELECT department,
           COUNT(*) AS employee_count,
           ROUND(AVG(salary), 2) AS average_salary,
           SUM(salary) AS total_salary
    FROM employees
    GROUP BY department
    HAVING COUNT(*) >= ?
    ORDER BY department
""", (2,))

for row in practice_cursor.fetchall():
    print(row)

('Engineering', 2, 77500.0, 155000)


## 7. Connect two tables with JOIN

A foreign key links each task to an employee. `JOIN` combines matching rows. Alex has two tasks; Nina has none. `INSERT OR IGNORE` skips duplicate task IDs if you rerun this cell.

In [8]:
practice_cursor.execute("""
    CREATE TABLE IF NOT EXISTS tasks (
        id INTEGER PRIMARY KEY,
        employee_id INTEGER NOT NULL,
        title TEXT NOT NULL,
        FOREIGN KEY (employee_id) REFERENCES employees(id)
    )
""")
practice_cursor.executemany(
    "INSERT OR IGNORE INTO tasks (id, employee_id, title) VALUES (?, ?, ?)",
    [
        (1, 1, "Automate a report"),
        (2, 1, "Review a script"),
        (3, 2, "Build an API"),
        (4, 3, "Resolve a support ticket"),
    ],
)
practice_conn.commit()

practice_cursor.execute("""
    SELECT e.name, t.title
    FROM employees AS e
    JOIN tasks AS t ON e.id = t.employee_id
    ORDER BY e.id, t.id
""")
for row in practice_cursor.fetchall():
    print(row)

('Alex', 'Automate a report')
('Alex', 'Review a script')
('Maya', 'Build an API')
('Sam', 'Resolve a support ticket')


## 8. Include employees with no tasks

`LEFT JOIN` keeps every employee even if no task matches. Use `COUNT(t.id)`, not `COUNT(*)`, so employees without tasks correctly get zero.

In [9]:
practice_cursor.execute("""
    SELECT e.name, COUNT(t.id) AS task_count
    FROM employees AS e
    LEFT JOIN tasks AS t ON e.id = t.employee_id
    GROUP BY e.id, e.name
    ORDER BY e.id
""")
for row in practice_cursor.fetchall():
    print(row)

('Alex', 2)
('Maya', 1)
('Sam', 1)
('Nina', 0)


## 9. Roll back a failed transaction

A transaction lets related changes succeed or fail together. The second insert below deliberately repeats employee ID 1. `rollback()` undoes the first insert too, so employee ID 90 is not kept. The error is expected and handled.

In [10]:
try:
    practice_cursor.execute(
        "INSERT INTO employees (id, name, department, salary) VALUES (?, ?, ?, ?)",
        (90, "Transaction Demo", "Support", 45000),
    )
    practice_cursor.execute(
        "INSERT INTO employees (id, name, department, salary) VALUES (?, ?, ?, ?)",
        (1, "Duplicate ID", "Support", 45000),
    )
    practice_conn.commit()
except sqlite3.IntegrityError as error:
    practice_conn.rollback()
    print("Transaction rolled back:", error)

practice_cursor.execute("SELECT * FROM employees WHERE id = ?", (90,))
print("Employee 90 after rollback:", practice_cursor.fetchone())

Transaction rolled back: UNIQUE constraint failed: employees.id
Employee 90 after rollback: None


## 10. Export query results as CSV text

Use Python's `csv` module to turn a SQL report into spreadsheet-friendly data. `StringIO` keeps the CSV in memory, so this example does not create or overwrite any files.

In [11]:
import csv
import io

practice_cursor.execute(
    "SELECT name, department, salary FROM employees ORDER BY id"
)
csv_buffer = io.StringIO(newline="")
csv_writer = csv.writer(csv_buffer)
csv_writer.writerow([column[0] for column in practice_cursor.description])
csv_writer.writerows(practice_cursor.fetchall())
employee_csv = csv_buffer.getvalue()
csv_buffer.close()
print(employee_csv)

name,department,salary
Alex,Engineering,75000
Maya,Engineering,80000
Sam,Support,50000
Nina,Sales,60000



## 11. Import CSV data into SQL

`DictReader` reads columns by name. CSV values start as strings, so convert quantities with `int()` before inserting. This small inventory table is separate from the employee data.

In [12]:
import csv
import io

inventory_csv = "item,quantity\nKeyboard,5\nMouse,10\nMonitor,3\n"
with io.StringIO(inventory_csv) as inventory_buffer:
    inventory_rows = [
        (row["item"], int(row["quantity"]))
        for row in csv.DictReader(inventory_buffer)
    ]

practice_cursor.execute("""
    CREATE TABLE IF NOT EXISTS inventory (
        item TEXT PRIMARY KEY,
        quantity INTEGER NOT NULL CHECK (quantity >= 0)
    )
""")
practice_cursor.executemany(
    "INSERT OR IGNORE INTO inventory (item, quantity) VALUES (?, ?)",
    inventory_rows,
)
practice_conn.commit()

practice_cursor.execute("SELECT * FROM inventory ORDER BY item")
for row in practice_cursor.fetchall():
    print(row)

('Keyboard', 5)
('Monitor', 3)
('Mouse', 10)


## 12. Put a SQL query inside a Python function

Reuse the same query with different departments. Pass the connection into the function rather than opening a new database each time. Notice the comma in `(department_name,)`: it makes a one-item tuple. No matches returns an empty list.

In [13]:
def find_employees_by_department(connection, department_name):
    result_cursor = connection.execute(
        "SELECT name, salary FROM employees WHERE department = ? ORDER BY name",
        (department_name,),
    )
    return result_cursor.fetchall()


print("Engineering:", find_employees_by_department(practice_conn, "Engineering"))
print("Support:", find_employees_by_department(practice_conn, "Support"))
print("No matches:", find_employees_by_department(practice_conn, "Unknown"))

Engineering: [('Alex', 75000), ('Maya', 80000)]
Support: [('Sam', 50000)]
No matches: []


## What else can you build with Python and SQL?

- **Expense tracker:** store dates, categories, and amounts in cents; calculate monthly totals with `SUM` and `GROUP BY`.
- **Inventory tracker:** add products, record stock changes, and find low-stock items with `WHERE`.
- **Task manager:** save tasks, due dates, and completion status; join tasks to their owners.
- **Network device inventory:** store device names, IP addresses, and check results; report devices whose latest check failed.
- **CSV reporting tool:** import a CSV, run summary queries, and export a report like Examples 10 and 11.

**Practice challenges:** find employees earning above the overall average using a subquery; show only employees with zero tasks; add a task status column and count completed tasks per employee.

## Close the practice connection

Run the next cell when finished. Closing this connection discards the in-memory practice data. To practice again afterward, rerun Example 1 and the following cells in order. Your original `database.db` remains unchanged.

In [14]:
practice_cursor.close()
practice_conn.close()
print("Practice connection closed.")

Practice connection closed.
